# 🧮 Math Chatbox Server - Kaggle
## Hướng dẫn sử dụng
1. Upload notebook này lên Kaggle
2. Bật GPU (nếu cần)
3. Chạy từng cell theo thứ tự
4. Copy link ngrok ở cell cuối để dùng cho web chatbox
5. Hỗ trợ 16 người dùng đồng thời với batch processing

## Cell 1: Cài đặt thư viện

In [ ]:
!pip install flask flask-cors pyngrok transformers torch accelerate sentencepiece protobuf -q
!pip install gunicorn gevent -q
print("✅ Đã cài đặt xong tất cả thư viện!")

## Cell 2: Cấu hình Model & Ngrok Token
### Danh sách model gợi ý:
- `Qwen/Qwen2.5-Math-1.5B-Instruct` (nhẹ, nhanh)
- `Qwen/Qwen2.5-Math-7B-Instruct` (tốt hơn, cần GPU)
- `microsoft/Phi-3-mini-4k-instruct` (đa năng)
- `deepseek-ai/deepseek-math-7b-instruct` (chuyên toán)

### Lấy Ngrok Token:
1. Đăng ký tại https://ngrok.com
2. Vào Dashboard → Your Authtoken
3. Copy và paste vào biến `NGROK_TOKEN` bên dưới

In [ ]:
#==========================================================
# ⚙️ CẤU HÌNH - THAY ĐỔI Ở ĐÂY
#==========================================================

# Danh sách model có thể chọn (thay đổi MODEL_INDEX để chọn model)
MODEL_LIST = [
    "Qwen/Qwen2.5-Math-1.5B-Instruct",    # 0 - Nhẹ, nhanh
    "Qwen/Qwen2.5-Math-7B-Instruct",       # 1 - Tốt, cần GPU
    "microsoft/Phi-3-mini-4k-instruct",     # 2 - Đa năng
    "deepseek-ai/deepseek-math-7b-instruct",# 3 - Chuyên toán
]

# Chọn model (thay đổi số 0 thành index model muốn dùng)
MODEL_INDEX = 0
SELECTED_MODEL = MODEL_LIST[MODEL_INDEX]

# Ngrok token - THAY BẰNG TOKEN CỦA BẠN
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"

# Cấu hình server
MAX_CONCURRENT_USERS = 16
BATCH_SIZE = 16
MAX_NEW_TOKENS = 2048
SERVER_PORT = 5000

print(f"📋 Model được chọn: {SELECTED_MODEL}")
print(f"👥 Số người dùng tối đa: {MAX_CONCURRENT_USERS}")
print(f"📦 Batch size: {BATCH_SIZE}")

## Cell 3: Load Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

print(f"🔄 Đang tải model: {SELECTED_MODEL}")
print(f"🖥️ GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

start_time = time.time()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    SELECTED_MODEL,
    trust_remote_code=True,
    padding_side='left'
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    SELECTED_MODEL,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

if not torch.cuda.is_available():
    model = model.to(device)

model.eval()

elapsed = time.time() - start_time
print(f"\n✅ Model đã sẵn sàng! (Thời gian tải: {elapsed:.1f}s)")
print(f"📍 Device: {device}")

## Cell 4: Tạo Server với Batch Processing

In [ ]:
import json
import threading
import queue
import uuid
import time
from flask import Flask, request, jsonify, Response
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

#==========================================================
# BATCH PROCESSING ENGINE
#==========================================================

# Queue để nhận request từ users
request_queue = queue.Queue()
# Dict lưu kết quả cho từng request
result_store = {}
# Lock cho thread safety
result_lock = threading.Lock()
# Đếm số user đang hoạt động
active_users = 0
active_users_lock = threading.Lock()
# Thống kê
stats = {
    "total_requests": 0,
    "total_batches": 0,
    "active_users": 0,
    "queue_size": 0
}

SYSTEM_PROMPT = """Bạn là trợ lý toán học thông minh. Hãy giải bài toán một cách chi tiết, rõ ràng.
- Trình bày lời giải từng bước
- Sử dụng ký hiệu toán học chuẩn (LaTeX khi cần)
- Giải thích logic từng bước
- Đưa ra đáp án cuối cùng rõ ràng
- Nếu có nhiều cách giải, trình bày cách tối ưu nhất
Trả lời bằng tiếng Việt."""

def format_prompt(message, history=None):
    """Format prompt cho model"""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    if history:
        for h in history[-4:]:  # Giữ 4 tin nhắn gần nhất
            messages.append({"role": "user", "content": h["user"]})
            messages.append({"role": "assistant", "content": h["assistant"]})
    
    messages.append({"role": "user", "content": message})
    
    try:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        # Fallback nếu model không hỗ trợ chat template
        prompt = f"System: {SYSTEM_PROMPT}\n\nUser: {message}\n\nAssistant:"
    
    return prompt


def batch_inference(prompts):
    """Chạy inference cho một batch prompts"""
    try:
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048
        ).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        # Decode chỉ phần generated
        results = []
        for i, output in enumerate(outputs):
            input_len = inputs["input_ids"][i].shape[0]
            generated = output[input_len:]
            text = tokenizer.decode(generated, skip_special_tokens=True)
            results.append(text.strip())
        
        return results
    except Exception as e:
        print(f"❌ Batch inference error: {e}")
        return [f"Lỗi xử lý: {str(e)}"] * len(prompts)


def batch_worker():
    """Worker thread xử lý batch requests"""
    print("🔄 Batch worker started...")
    while True:
        batch_items = []
        
        # Chờ ít nhất 1 request
        try:
            item = request_queue.get(timeout=1.0)
            batch_items.append(item)
        except queue.Empty:
            continue
        
        # Thu thập thêm requests trong 0.5s hoặc đủ batch size
        deadline = time.time() + 0.5
        while len(batch_items) < BATCH_SIZE and time.time() < deadline:
            try:
                item = request_queue.get(timeout=0.1)
                batch_items.append(item)
            except queue.Empty:
                break
        
        if not batch_items:
            continue
        
        # Xử lý batch
        batch_size = len(batch_items)
        stats["total_batches"] += 1
        stats["queue_size"] = request_queue.qsize()
        
        print(f"📦 Processing batch #{stats['total_batches']}: {batch_size} requests")
        
        prompts = [item["prompt"] for item in batch_items]
        request_ids = [item["id"] for item in batch_items]
        
        # Batch inference
        results = batch_inference(prompts)
        
        # Lưu kết quả
        with result_lock:
            for req_id, result in zip(request_ids, results):
                result_store[req_id] = {
                    "status": "done",
                    "response": result,
                    "timestamp": time.time()
                }
        
        print(f"✅ Batch #{stats['total_batches']} done: {batch_size} responses")


# Cleanup thread - xóa kết quả cũ
def cleanup_worker():
    while True:
        time.sleep(300)  # Mỗi 5 phút
        with result_lock:
            now = time.time()
            expired = [k for k, v in result_store.items() 
                      if now - v.get("timestamp", now) > 600]
            for k in expired:
                del result_store[k]
            if expired:
                print(f"🧹 Cleaned up {len(expired)} expired results")


#==========================================================
# API ROUTES
#==========================================================

@app.route("/")
def index():
    return jsonify({
        "status": "running",
        "model": SELECTED_MODEL,
        "max_users": MAX_CONCURRENT_USERS,
        "batch_size": BATCH_SIZE,
        "stats": stats
    })


@app.route("/api/chat", methods=["POST"])
def chat():
    """Submit a chat request"""
    global active_users
    
    data = request.json
    message = data.get("message", "").strip()
    history = data.get("history", [])
    
    if not message:
        return jsonify({"error": "Message is required"}), 400
    
    with active_users_lock:
        if active_users >= MAX_CONCURRENT_USERS:
            return jsonify({
                "error": "Server đang quá tải. Vui lòng thử lại sau.",
                "queue_full": True
            }), 429
        active_users += 1
        stats["active_users"] = active_users
    
    try:
        # Tạo prompt
        prompt = format_prompt(message, history)
        
        # Tạo request ID
        req_id = str(uuid.uuid4())
        stats["total_requests"] += 1
        
        # Đưa vào queue
        with result_lock:
            result_store[req_id] = {"status": "queued", "timestamp": time.time()}
        
        request_queue.put({
            "id": req_id,
            "prompt": prompt
        })
        
        # Chờ kết quả (polling)
        timeout = 120  # 2 phút timeout
        start = time.time()
        
        while time.time() - start < timeout:
            with result_lock:
                result = result_store.get(req_id, {})
                if result.get("status") == "done":
                    response_text = result["response"]
                    del result_store[req_id]
                    return jsonify({
                        "response": response_text,
                        "model": SELECTED_MODEL,
                        "request_id": req_id
                    })
            time.sleep(0.2)
        
        # Timeout
        with result_lock:
            if req_id in result_store:
                del result_store[req_id]
        
        return jsonify({"error": "Request timeout. Vui lòng thử lại."}), 504
    
    finally:
        with active_users_lock:
            active_users -= 1
            stats["active_users"] = active_users


@app.route("/api/models", methods=["GET"])
def get_models():
    """Trả về danh sách models"""
    return jsonify({
        "models": MODEL_LIST,
        "current": SELECTED_MODEL,
        "current_index": MODEL_INDEX
    })


@app.route("/api/stats", methods=["GET"])
def get_stats():
    """Trả về thống kê server"""
    stats["queue_size"] = request_queue.qsize()
    return jsonify(stats)


@app.route("/api/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model": SELECTED_MODEL})


print("✅ Server routes đã được cấu hình!")
print(f"📋 Endpoints:")
print(f"   POST /api/chat     - Gửi tin nhắn")
print(f"   GET  /api/models   - Danh sách models")
print(f"   GET  /api/stats    - Thống kê server")
print(f"   GET  /api/health   - Health check")

## Cell 5: Khởi động Server + Ngrok

In [ ]:
from pyngrok import ngrok, conf
import threading

# Cấu hình ngrok
ngrok.kill()  # Kill any existing tunnels
conf.get_default().auth_token = NGROK_TOKEN

# Start batch worker thread
batch_thread = threading.Thread(target=batch_worker, daemon=True)
batch_thread.start()

# Start cleanup worker thread
cleanup_thread = threading.Thread(target=cleanup_worker, daemon=True)
cleanup_thread.start()

# Start ngrok tunnel
public_url = ngrok.connect(SERVER_PORT, "http")

print("="*60)
print("🚀 SERVER ĐÃ SẴN SÀNG!")
print("="*60)
print(f"\n🌐 Public URL: {public_url}")
print(f"\n📋 Copy URL này vào file index.html")
print(f"   Thay thế giá trị API_URL = \"{public_url}\"")
print(f"\n🧮 Model: {SELECTED_MODEL}")
print(f"👥 Max users: {MAX_CONCURRENT_USERS}")
print(f"📦 Batch size: {BATCH_SIZE}")
print("="*60)

# Chạy Flask server (blocking)
app.run(host="0.0.0.0", port=SERVER_PORT, threaded=True)